# TP 4 — Apprentissage supervisé 2A (correction)

**Deux parties :**
- **Partie 1 (TD)** : matrices de confusion et courbes ROC, à partir d'un tableau de scores fourni.
- **Partie 2 (TP)** : détection de **spam** (jeu `spambase`), comparaison de plusieurs modèles via **ROC / AUC**.

> Correction en **Python / scikit-learn**. Le jeu `spambase` est téléchargé depuis **OpenML**.


## Brief de démarrage — Les outils de ce TP

> **À lire avant de commencer.** Ce TP ne porte pas sur un nouveau modèle mais sur la façon d'**évaluer et de
> comparer** des classifieurs quand le seuil de décision compte (détection de spam). Voici les outils clés.

**1. Matrice de confusion — le point de départ.**
Pour un seuil donné, on croise vérité et prédiction :

| | prédit négatif | prédit positif |
|---|---|---|
| **réel négatif** | VN | FP |
| **réel positif** | FN | VP |

On en tire notamment :
- **Sensibilité / rappel** (TVP) $= \frac{VP}{VP+FN}$ : part des positifs bien détectés ;
- **Spécificité** $= \frac{VN}{VN+FP}$ ; **taux de faux positifs** (TFP) $= 1 - \text{spécificité}$ ;
- **Précision** $= \frac{VP}{VP+FP}$.

**2. Courbe ROC et AUC — évaluer *indépendamment du seuil*.**
Un classifieur produit un **score** $s(x)$. En faisant **varier le seuil**, on parcourt tous les compromis
(TVP en fonction du TFP) : c'est la **courbe ROC**. L'**AUC** (aire sous la courbe) résume la capacité du modèle
à **ordonner** les positifs avant les négatifs : $0.5$ = aléatoire, $1$ = parfait. L'AUC ne dépend **pas** du
seuil choisi, ce qui la rend idéale pour **comparer des modèles**.

**3. Coûts asymétriques et choix du seuil.**
En pratique, une erreur peut coûter plus cher qu'une autre (laisser passer un spam vs bloquer un vrai message).
On **déplace le seuil** (ou on pondère les classes) pour privilégier la sensibilité ou la spécificité selon le
coût métier. La courbe ROC aide à **choisir le point de fonctionnement**.

> **Méthode du TP.** (Partie 1) construire matrices de confusion et courbe ROC à la main depuis des scores
> fournis ; (Partie 2) entraîner plusieurs classifieurs (LDA/QDA, CART, forêt, k-NN…) sur `spambase` et les
> **comparer via ROC / AUC**.

In [1]:
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import fetch_openml
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

sys.path += [".", ".."]          # plot_theme.py est a la racine du projet
from plot_theme import *         # couleurs, pick_color(), theme sombre applique par defaut

SEED = 0
np.random.seed(SEED)             # graine unique posee une fois : decoupage, arbres, forets reproductibles

## Partie 1 — Matrices de confusion, courbes ROC

### Rappel de cours

On est en discrimination binaire, l'événement d'intérêt est $Y = 1$. Un classifieur produit un **score**
$s(x) \in [0,1]$ (estimation de $\mathbb{P}(Y=1\mid x)$) ; on décide $\hat{Y} = 1$ si $s(x) \ge$ **seuil**.

**Matrice de confusion** (à seuil fixé) :

|               | Prédit 1 | Prédit 0 |
|---------------|----------|----------|
| **Réel 1**    | VP (vrai positif) | FN (faux négatif) |
| **Réel 0**    | FP (faux positif) | VN (vrai négatif) |

On en déduit :
- **Sensibilité** (taux de vrais positifs, TPR) : $\text{TPR} = \dfrac{VP}{VP + FN}$ ;
- **Spécificité** : $\dfrac{VN}{VN + FP}$, et **taux de faux positifs** $\text{FPR} = 1 - \text{spécificité} = \dfrac{FP}{FP + VN}$.

**Courbe ROC** : on fait **varier le seuil** de $1$ vers $0$ et on trace le point $(\text{FPR}, \text{TPR})$.
Elle décrit le compromis entre détecter les positifs et éviter les fausses alertes, **indépendamment du seuil**.


### Question 1 — Construction de la matrice de confusion

À seuil fixé, on compare la prédiction $\hat{Y} = \mathbb{1}\{s(x) \ge \text{seuil}\}$ à la vérité $Y$, et on
compte les 4 cas (VP, FP, VN, FN) dans le tableau ci-dessus. Chaque seuil donne **une** matrice de confusion.


### Question 2 — Données fournies et tracé de la ROC

**2.1** Le **score** est la probabilité estimée $\hat{\mathbb{P}}(Y=1\mid x)$ donnée par le classifieur.
La colonne **« Individual »** est l'**identifiant** de l'observation ; la colonne **« Observation Y »** est la
**vraie classe** ($0$ ou $1$).

On saisit les 27 observations fournies dans l'énoncé.


In [2]:
donnees = pd.DataFrame({
    "Individual": range(1, 28),
    "Y": [1,0,1,1,1,1,0,0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0],
    "Score": [0.92825,0.82762,0.74119,0.73182,0.72418,0.63537,0.63155,0.61763,
              0.49700,0.46686,0.37062,0.31244,0.29258,0.26446,0.22945,0.21983,
              0.17818,0.13490,0.12012,0.05337,0.01124,0.00639,0.00241,0.00102,
              0.00099,0.00040,0.00011],
})
P = int((donnees.Y == 1).sum())   # positifs
N = int((donnees.Y == 0).sum())   # negatifs
print(f"Nombre de positifs P = {P} | negatifs N = {N}")
donnees.head()


Nombre de positifs P = 9 | negatifs N = 18


,Individual,Y,Score
0,1,1,0.92825
1,2,0,0.82762
2,3,1,0.74119
3,4,1,0.73182
4,5,1,0.72418


**2.2** Sur la courbe ROC : l'**axe des x** est le **taux de faux positifs** (FPR = $1 -$ spécificité),
l'**axe des y** est le **taux de vrais positifs** (TPR = sensibilité). On la trace en **abaissant le seuil**
progressivement : à chaque score utilisé comme seuil, on calcule (FPR, TPR) et on relie les points.

**2.3** On trie par score décroissant (déjà le cas) et on ajoute les colonnes cumulées **VP** et **FP**, d'où
**TPR** et **FPR**.


In [3]:
tab = donnees.sort_values("Score", ascending=False).reset_index(drop=True)
tab["VP_cumule"] = (tab.Y == 1).cumsum()      # vrais positifs si seuil = ce score
tab["FP_cumule"] = (tab.Y == 0).cumsum()      # faux positifs
tab["TPR"] = tab["VP_cumule"] / P
tab["FPR"] = tab["FP_cumule"] / N
tab[["Individual", "Y", "Score", "VP_cumule", "FP_cumule", "TPR", "FPR"]]


,Individual,Y,Score,VP_cumule,FP_cumule,TPR,FPR
0,1,1,0.92825,1,0,0.111111,0.000000
1,2,0,0.82762,1,1,0.111111,0.055556
2,3,1,0.74119,2,1,0.222222,0.055556
3,4,1,0.73182,3,1,0.333333,0.055556
4,5,1,0.72418,4,1,0.444444,0.055556
5,6,1,0.63537,5,1,0.555556,0.055556
6,7,0,0.63155,5,2,0.555556,0.111111
7,8,0,0.61763,5,3,0.555556,0.166667
8,9,1,0.49700,6,3,0.666667,0.166667
9,10,1,0.46686,7,3,0.777778,0.166667


In [4]:
# On part du point (0,0) : seuil > au plus grand score => rien n'est predit positif
fpr = np.r_[0, tab["FPR"]]
tpr = np.r_[0, tab["TPR"]]
roc_evalue = go.Scatter(x=fpr, y=tpr, mode="lines+markers", line_shape="hv", name="Modele evalue",
                        line=dict(width=3, color=pick_color(0)), marker=dict(size=8, color=pick_color(2)))

fig = go.Figure(roc_evalue)
fig.update_layout(title="Courbe ROC", xaxis_title="Taux de faux positifs (FPR)",
                  yaxis_title="Taux de vrais positifs (TPR)")
fig.show()

**2.4** On ajoute :
- le **modèle parfait** : il classe tous les positifs avant tous les négatifs → la courbe monte à
  $(0,1)$ puis va à $(1,1)$ (coin supérieur gauche) ;
- le **modèle aléatoire** : la **diagonale** $y = x$ (AUC $= 0.5$).


In [5]:
fig = go.Figure(roc_evalue)
fig.add_trace(go.Scatter(x=[0, 0, 1], y=[0, 1, 1], mode="lines", name="Modele parfait",
                         line=dict(width=3, color=pick_color(3))))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Modele aleatoire",
                         line=dict(color=MUTED, dash="dash")))
fig.update_layout(title="Courbe ROC : modele evalue, parfait et aleatoire", xaxis_title="FPR", yaxis_title="TPR")
fig.show()

print("AUC du modele evalue :", round(roc_auc_score(donnees.Y, donnees.Score), 3))

AUC du modele evalue : 0.883


### Question 3 — Analyse de la courbe ROC

**3.1 Points les plus intéressants.** Ceux du **coin supérieur gauche** (FPR faible, TPR élevé) : ils offrent
le meilleur compromis « détecter beaucoup de positifs / peu de fausses alertes ». On choisit souvent le seuil
qui **maximise** $\text{TPR} - \text{FPR}$ (indice de **Youden**), c'est-à-dire le point le plus éloigné de la
diagonale.

**3.2 Aire sous la courbe (AUC).** Elle représente la **probabilité que le modèle attribue un score plus élevé
à un positif tiré au hasard qu'à un négatif tiré au hasard** : c'est une mesure de la **qualité de
l'ordonnancement**. Valeurs : entre **0.5** (modèle aléatoire) et **1** (modèle parfait) ; $< 0.5$ signifierait
un modèle « à l'envers ».

**3.3 Comparer deux modèles.** On compare leurs **AUC** : le modèle d'AUC la plus grande est globalement
meilleur. Idéalement, une courbe qui **domine** l'autre partout (toujours au-dessus) est préférable quel que
soit le seuil ; si les courbes se croisent, le meilleur choix dépend du seuil / du coût des erreurs visé.


## Partie 2 — Détection de spam

### Rappel de cours — Comparaison de modèles et coûts différenciés

On implémente plusieurs classifieurs et on les compare par **ROC / AUC** (métrique de séparation indépendante
du seuil). Un point spécifique ici : le **coût différencié** (5 pour 1). Toutes les erreurs ne se valent pas
(laisser passer un spam vs bloquer un email légitime). On l'intègre via des **poids de classe**
(`class_weight`) : donner un poids 5× à une classe revient à pénaliser 5× plus les erreurs sur cette classe.

Le jeu `spambase` : **4601** emails, **57** variables (fréquences de mots/caractères, longueurs de séquences de
majuscules), cible **`spam`** (1) / non-spam (0).


In [6]:
spam = fetch_openml("spambase", version=1, as_frame=True)
X = spam.data
y = spam.target.astype(int)
print("Dimensions :", X.shape, "| proportion de spam :", round(y.mean(), 3))

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y
)
print("Apprentissage :", X_tr.shape, "| Test :", X_te.shape)

Dimensions : (4601, 57) | proportion de spam : 0.394


Apprentissage : (3450, 57) | Test : (1151, 57)


### Observation des données

Avant d'entraîner le moindre modèle, on regarde les données : **variable cible** (équilibre des classes),
**distribution** des variables explicatives et **corrélations**. Ce sont ces constats qui justifient les
choix de prétraitement et de métriques qui suivent.

#### 1. La variable cible

In [7]:
effectifs = y.value_counts().rename({0: "non-spam", 1: "spam"})
print(pd.DataFrame({"Effectif": effectifs, "Proportion": (effectifs / len(y)).round(3)}))
print("Accuracy du classifieur 'toujours non-spam' :", round(1 - y.mean(), 3))

fig = go.Figure(go.Bar(x=effectifs.index, y=effectifs.values, marker_color=[pick_color(0), pick_color(1)]))
fig.update_layout(title="Repartition de la variable cible", yaxis_title="Nombre d'emails")
fig.show()

          Effectif  Proportion
class                         
non-spam      2788       0.606
spam          1813       0.394
Accuracy du classifieur 'toujours non-spam' : 0.606


**Lecture.** Environ **39 % de spams** pour 61 % d'emails légitimes : les classes sont **déséquilibrées sans
être extrêmes**. Deux conséquences :
- un classifieur « idiot » qui répond toujours *non-spam* a déjà une **accuracy de 0.606** : c'est la **référence
  minimale** à battre, et la raison pour laquelle l'accuracy seule est trompeuse ;
- on **stratifie** le découpage apprentissage / test (et les plis de validation croisée) pour conserver cette
  proportion partout.

#### 2. Les variables explicatives

Les 57 variables sont de trois types : **48 fréquences de mots** (`make`, `free`, `hp`…, en % des mots de
l'email), **6 fréquences de caractères** (`;`, `(`, `[`, `!`, `$`, `#`) et **3 longueurs de séquences de
majuscules** (moyenne, plus longue, total).

In [8]:
print("Valeurs manquantes :", X.isna().sum().sum())
resume = pd.DataFrame({
    "% de zeros": (X == 0).mean() * 100,
    "mediane": X.median(),
    "max": X.max(),
    "asymetrie": X.skew(),
}).round(2)
print("Mediane du % de zeros :", resume["% de zeros"].median())
resume.sort_values("asymetrie", ascending=False).head(10)

Valeurs manquantes : 0
Mediane du % de zeros : 83.7


,% de zeros,mediane,max,asymetrie
char_freq_%23,83.70,0.00,19.83,31.06
capital_run_length_longest,0.00,15.00,9989.00,30.76
word_freq_parts,98.20,0.00,8.33,28.26
word_freq_3d,98.98,0.00,42.81,26.23
capital_run_length_average,0.00,2.28,1102.50,23.76
char_freq_%5B,88.50,0.00,4.08,21.08
word_freq_table,98.63,0.00,2.17,19.87
word_freq_conference,95.59,0.00,10.00,19.72
word_freq_project,92.89,0.00,20.00,18.77
char_freq_%21,50.92,0.00,32.48,18.66


**Lecture.**
- **Aucune valeur manquante**, mais énormément de **zéros** : la plupart des mots sont absents de la plupart des
  emails (la médiane du % de zéros est d'environ 84 %), d'où des **médianes nulles**.
- Les distributions sont **très asymétriques** (coefficient d'asymétrie de 20 à 30) avec des **valeurs extrêmes**
  (une séquence de majuscules de près de 10 000 caractères).
- Les **échelles sont très différentes** : fréquences entre 0 et 100, longueurs de majuscules jusqu'à plusieurs
  milliers.

Conséquences : l'hypothèse **gaussienne** de LDA/QDA est clairement fausse ; le **k-NN** doit être précédé d'une
**standardisation** (sinon les longueurs de majuscules écrasent toutes les autres variables) ; les **arbres**,
eux, sont insensibles à l'échelle et aux valeurs extrêmes.

#### 3. Corrélations

**Avec la cible.** On calcule la corrélation (de Pearson) entre chaque variable et l'indicatrice « spam ».

In [9]:
corr_cible = X.corrwith(y).sort_values(key=abs, ascending=False)
top = corr_cible.head(10)
fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation="h", marker_color=pick_color(0)))
fig.update_layout(title="Top 10 des correlations avec la cible", xaxis_title="Correlation avec spam",
                  yaxis=dict(autorange="reversed"))
fig.show()

# Distribution par classe des 4 variables les plus liees a la cible (log(1+x) a cause de l'asymetrie)
fig = go.Figure()
for i, (classe, nom) in enumerate([(0, "non-spam"), (1, "spam")]):
    sous = np.log1p(X.loc[y == classe, top.index[:4]]).melt()
    fig.add_trace(go.Box(x=sous["variable"], y=sous["value"], name=nom, marker_color=pick_color(i)))
fig.update_layout(boxmode="group", title="Variables les plus liees a la cible, par classe",
                  yaxis_title="log(1 + x)")
fig.show()

**Lecture.** Les mots les plus liés au spam sont ceux du marketing (`your`, `000`, `remove`, `free`,
`business`) et le caractère `$`. À l'inverse, `hp` (Hewlett-Packard, où le corpus a été collecté) est un
marqueur d'email **légitime** (corrélation négative). Mais **aucune corrélation ne dépasse 0.4** : aucune variable
ne suffit seule, il faut **combiner** beaucoup de variables, ce que font les modèles multivariés.

⚠️ La corrélation de Pearson ne mesure qu'une liaison **linéaire** et est sensible aux valeurs extrêmes : sur des
variables aussi asymétriques, les boîtes à moustaches en échelle $\log(1+x)$ sont plus parlantes.

**Entre variables explicatives.** Des variables très corrélées entre elles sont **redondantes**.

In [10]:
corr = X.corr()
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.columns,
                           colorscale=DIVERGING_SCALE, zmin=-1, zmax=1))
fig.update_layout(title="Correlations entre les 57 variables", height=750)
fig.show()

# Paires les plus correlees (triangle superieur pour ne pas compter chaque paire deux fois)
paires = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
paires.sort_values(key=abs, ascending=False).head(8).round(3)

word_freq_857     word_freq_415           0.996
                  word_freq_direct        0.848
word_freq_415     word_freq_direct        0.845
word_freq_telnet  word_freq_857           0.738
                  word_freq_415           0.735
word_freq_857     word_freq_technology    0.730
word_freq_415     word_freq_technology    0.727
word_freq_telnet  word_freq_direct        0.700
dtype: float64

**Lecture.** La plupart des corrélations sont faibles, mais un **bloc** de variables est fortement lié : `857`,
`415`, `direct`, `telnet`, `technology` (en particulier $r \approx 0.996$ entre `857` et `415`, quasiment la
même variable). Ce sont des numéros de téléphone / termes techniques présents dans les mêmes emails internes.

Impact sur les modèles :
- **LDA** : covariance quasi singulière → coefficients **instables et non interprétables** (la prédiction reste
  correcte) ;
- **QDA** : une covariance **par classe** à inverser, encore plus fragile → **régularisation** nécessaire ;
- **k-NN** : une information redondante **compte plusieurs fois** dans la distance ;
- **arbres / forêts** : peu gênés pour prédire, mais l'**importance** se répartit entre variables corrélées.

### Rappel — Points d'attention pour chaque modèle

| Modèle | Principe | Points d'attention (en particulier sur `spambase`) |
|---|---|---|
| **LDA** | Gaussiennes de **même** covariance → frontière **linéaire** | Hypothèse gaussienne fausse ici (zéros, asymétrie) ; sensible aux **valeurs extrêmes** et à la **colinéarité** (`857`/`415`) ; pas d'hyperparamètre ; invariante à l'échelle (pas besoin de standardiser). |
| **QDA** | Une covariance **par classe** → frontière **quadratique** | ≈ 1 650 paramètres de covariance **par classe** : matrices quasi singulières (variables presque toujours nulles dans une classe) → **régulariser** ; probabilités très extrêmes (0 ou 1) → mal calibrées, accuracy parfois décevante malgré une AUC correcte. |
| **CART** | Découpages successifs par seuils sur une variable | **Surapprentissage** si l'arbre est trop profond → régler profondeur / taille minimale des feuilles / `cp` par **validation croisée** ; très **instable** ; insensible à l'échelle et aux valeurs extrêmes (aucun prétraitement) ; probabilités = proportions dans les feuilles → peu de valeurs distinctes, ROC « en escalier ». |
| **CART coût 5:1** | Même arbre, erreurs sur les spams pénalisées 5× | Modifie l'**arbre lui-même** et le point de fonctionnement, pas la capacité à ordonner → l'évaluer sur le **rappel / le coût**, pas sur l'accuracy ; il « sacrifie » volontairement de l'accuracy. |
| **Forêt aléatoire** | Moyenne de nombreux arbres décorrélés (bootstrap + tirage de variables) | Hyperparamètre principal : nombre de variables tirées (`max_features` / `mtry`) ; **plus d'arbres** ne fait pas surapprendre (juste plus lent) ; probabilités « tassées » vers 0.5 → seuil théorique à manier avec prudence ; importances partagées entre variables corrélées ; peu interprétable. |
| **k-NN** | Vote des $k$ plus proches voisins | **Standardisation obligatoire** (échelles de 0–100 à plusieurs milliers), apprise sur le **train uniquement** (`Pipeline` / `preProcess`) ; $k$ par validation croisée ; 57 dimensions → **fléau de la dimension** ; variables redondantes comptées plusieurs fois ; probabilités multiples de $1/k$ ; prédiction coûteuse. |

**Pour tous les modèles :** les hyperparamètres se règlent par **validation croisée sur l'apprentissage** ; le jeu
de **test** ne sert qu'**une fois**, à la fin, pour comparer.

On construit les modèles demandés. Les hyperparamètres sensibles (profondeur d'arbre, `mtry`, `k`) sont
réglés par **validation croisée** ; LDA/QDA n'en ont quasiment pas. Le **k-NN** est précédé d'une
**standardisation** (`Pipeline`). Le **CART à coût différencié 5:1** utilise `class_weight={0:1, 1:5}`
(on pénalise 5× plus le fait de **rater un spam**).


In [11]:
def regle(estimateur, grille):
    """Regle les hyperparametres par validation croisee (AUC) et renvoie le meilleur modele."""
    return GridSearchCV(estimateur, grille, cv=5, scoring="roc_auc").fit(X_tr, y_tr).best_estimator_


grille_cart = {"max_depth": [3, 5, 7, 10, None], "min_samples_leaf": [1, 5, 10]}
modeles = {
    "LDA": LinearDiscriminantAnalysis().fit(X_tr, y_tr),
    "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1).fit(X_tr, y_tr),
    "CART": regle(DecisionTreeClassifier(), grille_cart),
    "CART cout 5:1": regle(DecisionTreeClassifier(class_weight={0: 1, 1: 5}), grille_cart),
    "Foret aleatoire": regle(RandomForestClassifier(n_estimators=300, n_jobs=-1),
                             {"max_features": ["sqrt", 0.3, 0.5], "min_samples_leaf": [1, 3]}),
    "k-NN": regle(Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())]),
                  {"knn__n_neighbors": list(range(3, 22, 2))}),
}
print("Modeles entraines :", list(modeles))

Modeles entraines : ['LDA', 'QDA', 'CART', 'CART cout 5:1', 'Foret aleatoire', 'k-NN']


### Comparaison finale sur le jeu de test : ROC, AUC et autres métriques

In [12]:
resultats = []
scores = {}
fig = go.Figure()
for i, (nom, est) in enumerate(modeles.items()):
    scores[nom] = est.predict_proba(X_te)[:, 1]
    pred = est.predict(X_te)
    auc = roc_auc_score(y_te, scores[nom])
    resultats.append({"Modele": nom, "Accuracy": accuracy_score(y_te, pred),
                      "Precision": precision_score(y_te, pred), "Rappel": recall_score(y_te, pred),
                      "F1": f1_score(y_te, pred), "AUC": auc})
    fpr, tpr, _ = roc_curve(y_te, scores[nom])
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{nom} (AUC={auc:.3f})",
                             line=dict(width=3, color=pick_color(i))))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatoire", line=dict(color=MUTED, dash="dash")))
fig.update_layout(title="Detection de spam : comparaison des modeles (ROC, test)",
                  xaxis_title="Taux de faux positifs", yaxis_title="Taux de vrais positifs")
fig.show()

comparaison = pd.DataFrame(resultats).sort_values("AUC", ascending=False).reset_index(drop=True).round(3)
comparaison

,Modele,Accuracy,Precision,Rappel,F1,AUC
0,Foret aleatoire,0.961,0.951,0.949,0.950,0.989
1,k-NN,0.911,0.911,0.859,0.884,0.967
2,LDA,0.898,0.920,0.813,0.863,0.963
3,CART,0.919,0.915,0.877,0.895,0.957
4,CART cout 5:1,0.874,0.773,0.963,0.858,0.954
5,QDA,0.856,0.745,0.965,0.841,0.952


### Comparaison des métriques : accuracy, F1 et AUC

| Métrique | Formule | Dépend du seuil ? | Ce qu'elle mesure | Limites |
|---|---|---|---|---|
| **Accuracy** | $\dfrac{VP + VN}{n}$ | oui | proportion de bonnes décisions | dépend de la **proportion de classes** (0.606 sans rien faire ici) ; traite FP et FN de la même façon |
| **F1** | $\dfrac{2\,\text{Précision}\cdot\text{Rappel}}{\text{Précision}+\text{Rappel}} = \dfrac{2VP}{2VP + FP + FN}$ | oui | qualité de détection de la **classe positive** (spam) | ignore les VN ; dépend de la classe choisie comme positive ; pondère précision et rappel à égalité (pas de coût 5:1) |
| **AUC** | $\mathbb{P}\big(s(X^+) > s(X^-)\big)$ | **non** | qualité de l'**ordonnancement** des scores | ne dit pas quel seuil choisir ; insensible à la calibration des probabilités ; ignore les coûts |

In [13]:
fig = go.Figure()
for i, metrique in enumerate(["Accuracy", "F1", "AUC"]):
    fig.add_trace(go.Bar(x=comparaison["Modele"], y=comparaison[metrique], name=metrique,
                         marker_color=pick_color(i)))
fig.add_hline(y=1 - y_te.mean(), line_dash="dash", line_color=MUTED, annotation_text="'toujours non-spam'")
fig.update_layout(barmode="group", title="Accuracy, F1 et AUC par modele (test)", yaxis=dict(range=[0.5, 1]))
fig.show()

**Lecture.**
- Tous les modèles battent nettement la référence « toujours non-spam » (0.606), et la **forêt aléatoire** est
  la meilleure sur les trois métriques.
- Les **classements** par accuracy / F1 et par AUC ne coïncident **pas** : le **CART** a la 2ᵉ meilleure accuracy
  et le 2ᵉ meilleur F1, mais une AUC **inférieure à celle de la LDA**. Ses probabilités ne prennent que quelques
  valeurs (une par feuille) : il **décide** bien au seuil 1/2 mais **ordonne** mal les emails entre eux.
- La **QDA** a un rappel très élevé (≈ 0.97) mais une précision faible (≈ 0.73) : elle classe trop d'emails en
  spam car ses probabilités sont extrêmes et mal calibrées ; son AUC reste correcte car l'**ordre** des scores
  est bon.
- Le **CART coût 5:1** fait exactement ce qu'on lui demande : rappel ↑, précision ↓, accuracy ↓. Aucune de ces
  trois métriques n'intègre le coût 5:1 : c'est l'objet du complément plus bas.
- Le **F1** est plus sévère que l'accuracy car il ne profite pas des nombreux VN (emails légitimes faciles).

**Influence du seuil.** Pour la forêt aléatoire, on fait varier le seuil de décision : accuracy et F1 bougent,
l'AUC reste fixe (elle ne dépend que de l'ordre des scores).

In [14]:
p_rf = scores["Foret aleatoire"]
seuils = np.linspace(0.05, 0.95, 19)
fig = go.Figure()
fig.add_trace(go.Scatter(x=seuils, y=[accuracy_score(y_te, p_rf >= s) for s in seuils], name="Accuracy",
                         line=dict(width=3, color=pick_color(0))))
fig.add_trace(go.Scatter(x=seuils, y=[f1_score(y_te, p_rf >= s) for s in seuils], name="F1",
                         line=dict(width=3, color=pick_color(1))))
fig.add_hline(y=roc_auc_score(y_te, p_rf), line_dash="dash", line_color=HIGHLIGHT, annotation_text="AUC")
fig.update_layout(title="Foret aleatoire : metriques selon le seuil de decision",
                  xaxis_title="Seuil sur P(spam | x)", yaxis_title="Valeur de la metrique")
fig.show()

**Lecture.** L'accuracy et le F1 **changent avec le seuil** (et ne sont pas forcément maximaux au même
endroit) ; l'AUC, elle, est une ligne horizontale. D'où la démarche : on **compare les modèles** avec l'AUC, puis
on **choisit le seuil** du modèle retenu selon le critère métier (accuracy, F1, ou coût $FP + 5\,FN$), sur des
données de validation.

### Complément — Coût différencié : matrices de confusion et choix du seuil

L'AUC compare les modèles **indépendamment du seuil**. Pour une **décision**, il faut fixer un seuil et regarder
la matrice de confusion. Avec un coût de 5 pour un spam raté (FN) et 1 pour un email légitime bloqué (FP), le
seuil optimal théorique est $s^\star = 1/6$ (voir l'annexe). On compare le **coût total** $FP + 5\,FN$ sur le
test pour plusieurs situations.

In [15]:
C_FP, C_FN = 1, 5
situations = {
    "CART (seuil 1/2)": scores["CART"] >= 0.5,
    "CART cout 5:1 (seuil 1/2)": scores["CART cout 5:1"] >= 0.5,
    "Foret (seuil 1/2)": scores["Foret aleatoire"] >= 0.5,
    "Foret (seuil 1/6)": scores["Foret aleatoire"] >= C_FP / (C_FP + C_FN),
}
lignes = []
for nom, pred in situations.items():
    vn, fp, fn, vp = confusion_matrix(y_te, pred).ravel()
    lignes.append({"Situation": nom, "VN": vn, "FP": fp, "FN": fn, "VP": vp,
                   "Sensibilite": vp / (vp + fn), "Specificite": vn / (vn + fp), "Cout FP + 5 FN": C_FP * fp + C_FN * fn})
display(pd.DataFrame(lignes).round(3))

# Cout total de la foret selon le seuil
seuils = np.linspace(0.01, 0.99, 99)
couts = [C_FP * np.sum((scores["Foret aleatoire"] >= s) & (y_te == 0))
         + C_FN * np.sum((scores["Foret aleatoire"] < s) & (y_te == 1)) for s in seuils]
fig = go.Figure(go.Scatter(x=seuils, y=couts, mode="lines", line=dict(width=3, color=pick_color(0)),
                           name="Cout total (test)"))
fig.add_vline(x=1 / 6, line_dash="dash", line_color=HIGHLIGHT, annotation_text="s* = 1/6")
fig.add_vline(x=0.5, line_dash="dot", line_color=MUTED, annotation_text="seuil 1/2")
fig.update_layout(title="Foret aleatoire : cout FP + 5 FN selon le seuil de decision",
                  xaxis_title="Seuil sur P(spam | x)", yaxis_title="Cout total sur le test")
fig.show()

,Situation,VN,FP,FN,VP,Sensibilite,Specificite,Cout FP + 5 FN
0,CART (seuil 1/2),656,41,49,405,0.892,0.941,286
1,CART cout 5:1 (seuil 1/2),568,129,16,438,0.965,0.815,209
2,Foret (seuil 1/2),675,22,23,431,0.949,0.968,137
3,Foret (seuil 1/6),576,121,4,450,0.991,0.826,141


**Lecture.** Au seuil $1/2$, un modèle minimise le **nombre** d'erreurs, pas leur **coût**. Déplacer la décision
vers les spams (CART pondéré 5:1, ou seuil abaissé) augmente la **sensibilité** (FN ↓) au prix de fausses
alertes supplémentaires (spécificité ↓). Le tableau et la courbe montrent deux choses :
- pour un modèle **peu performant** (CART), la pondération 5:1 réduit fortement le coût : on échange beaucoup de
  FN coûteux contre des FP bon marché ;
- pour la **forêt**, déjà très bonne au seuil $1/2$, le seuil théorique $s^\star = 1/6$ n'est pas forcément
  meilleur : il suppose des probabilités **bien calibrées** ($\hat{\mathbb{P}}(Y=1\mid x) \approx \eta(x)$), ce qui
  n'est pas le cas des forêts (leurs probabilités sont « tassées » vers le centre). La courbe de coût est plate
  sur une large plage : le choix du seuil compte moins que la qualité du score.

⚠️ En pratique, le seuil doit être choisi sur des données de **validation** (ou par validation croisée), jamais
sur le test : ici la courbe sur le test sert uniquement d'**illustration**.

### Conclusion

- La **forêt aléatoire** obtient généralement la **meilleure AUC** sur `spambase` (elle capture des
  interactions complexes tout en restant robuste). Le **CART** simple est en retrait (instable), et le
  **k-NN** est correct après standardisation.
- **LDA/QDA** donnent une bonne base linéaire/quadratique mais sont limitées par leurs hypothèses gaussiennes,
  peu réalistes sur des fréquences de mots très asymétriques.
- Le **CART à coût 5:1** ne vise pas la meilleure AUC : il **déplace le compromis** pour réduire un type
  d'erreur précis (ici rater un spam), ce qui se voit sur la forme de sa courbe ROC / sa matrice de confusion
  plutôt que sur l'AUC globale.
- On compare **objectivement** les modèles par leur **AUC** ; le choix final du **seuil** dépend ensuite du
  **coût** relatif des faux positifs (bloquer un email légitime) et des faux négatifs (laisser passer un spam).


## Annexe — Pour aller plus loin sur les matrices de confusion, ROC et coût différencié

### 1. La matrice de confusion n’est qu’un seuil donné
La confusion résume les erreurs pour un **seuil précis**. Si on change le seuil, les valeurs de VP, FP, VN et FN changent. La ROC permet de voir ce qui se passe pour tous les seuils à la fois.

### 2. Pourquoi la ROC est utile ?
La courbe ROC trace le compromis entre :
- **TPR** = sensibilité = proportion de positifs bien détectés ;
- **FPR** = proportion de négatifs mal classés.

Un bon modèle est un modèle dont la courbe est proche du coin supérieur gauche : il détecte beaucoup de vrais positifs sans trop de faux positifs.

### 3. Qu’est-ce que l’AUC ?
L’AUC mesure la probabilité que, pour un positif et un négatif choisis au hasard, le score du positif soit plus élevé que celui du négatif.

- **AUC = 0.5** : performance aléatoire ;
- **AUC = 1** : modèle parfait ;
- **AUC proche de 1** : bon pouvoir de classement.

Autrement dit, l’AUC dit si le modèle classe bien les individus dans l’ordre correct, indépendamment du seuil retenu.

### 4. Pourquoi le coût différencié change le choix du seuil ?
Dans un problème de spam, un faux positif (bloquer un email légitime) et un faux négatif (laisser passer un spam) n’ont pas le même coût. On peut donc choisir un seuil qui favorise :
- la détection des spam (si le coût des faux négatifs est élevé) ;
- la protection des emails légitimes (si le coût des faux positifs est élevé).

La courbe ROC décrit le compromis, mais le seuil final dépend du contexte économique ou métier.

**Conclusion.** La ROC permet de comparer les modèles indépendamment du seuil, tandis que la matrice de confusion et le seuil choisi rendent le compromis explicite. C’est exactement la bonne distinction à faire entre « performance globale » et « décision opérationnelle ».


---

## Annexe — Seuil optimal quand les erreurs n'ont pas le même coût

On note $c_{FN}$ le coût d'un faux négatif (spam non détecté) et $c_{FP}$ celui d'un faux positif (email
légitime bloqué). En un point $x$, avec $\eta(x) = \mathbb{P}(Y = 1 \mid X = x)$ :
- prédire **spam** coûte en moyenne $c_{FP}\,(1 - \eta(x))$ (on se trompe si c'est un email légitime) ;
- prédire **non-spam** coûte en moyenne $c_{FN}\,\eta(x)$ (on se trompe si c'est un spam).

La règle de Bayes **à coûts** prédit donc spam dès que $c_{FN}\,\eta(x) \ge c_{FP}\,(1 - \eta(x))$, soit
$$\eta(x) \;\ge\; s^\star = \frac{c_{FP}}{c_{FP} + c_{FN}}.$$
- Coûts égaux : $s^\star = 1/2$ (on retrouve la règle de Bayes du TP1) ;
- coût 5:1 (rater un spam coûte 5 fois plus) : $s^\star = 1/(1+5) = 1/6$.

**Deux façons équivalentes** d'intégrer ce coût : (i) garder un modèle standard et **abaisser le seuil** à
$1/6$ sur ses probabilités ; (ii) **pondérer** les classes à l'apprentissage (`class_weight`, matrice `loss`
de `rpart`), ce qui déplace directement la décision. Dans les deux cas, l'AUC (qui ne dépend pas du seuil)
change peu : c'est le **point de fonctionnement** sur la courbe ROC qui se déplace.